<a href="https://colab.research.google.com/github/phillipsjackson/directed-research/blob/main/Pokeropen_spiel/colabs/install_open_spiel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title ##### License { display-mode: "form" }
# Copyright 2019 DeepMind Technologies Ltd. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# OpenSpiel

* This Colab gets you started with installing OpenSpiel and its dependencies.
* OpenSpiel is a framework for reinforcement learning in games.
* The instructions are adapted from [here](https://github.com/deepmind/open_spiel/blob/master/docs/install.md).

## Install

Install OpenSpiel via pip:


In [2]:
%pip install --upgrade open_spiel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 8.4 MB/s eta 0:00:00


# It's play time!

In [3]:
import numpy as np
import pyspiel

game = pyspiel.load_game("tic_tac_toe")
state = game.new_initial_state()

while not state.is_terminal():
  state.apply_action(np.random.choice(state.legal_actions()))
  print(str(state) + "\n")

x..
...
...

x.o
...
...

x.o
x..
...

x.o
xo.
...

x.o
xo.
x..



In [4]:
import random
import pyspiel
from open_spiel.python.algorithms import external_sampling_mccfr

# Mappings for human readability
CARD_NAMES = {
    0: "Jack ♠", 1: "Jack ♥", 2: "Jack ♦",
    3: "Queen ♠", 4: "Queen ♥", 5: "Queen ♦",
    6: "King ♠", 7: "King ♥", 8: "King ♦"
}
ACTION_NAMES = {0: "Fold", 1: "Call / Check", 2: "Raise"}

# 1. Load 3-Player Leduc Poker
game = pyspiel.load_game("leduc_poker(players=3)")

# 2. Initialize Fast MCCFR Solver
# Full sampling ensures we compute a true average policy across iterations
mccfr_solver = external_sampling_mccfr.ExternalSamplingSolver(
    game,
    external_sampling_mccfr.AverageType.FULL
)

print("Training 3-player MCCFR over 5,000 iterations (takes ~2-3 seconds)...")
for i in range(1, 300):
    mccfr_solver.iteration()
    if i % 1000 == 0:
        print(f"  -> Completed {i}/5000 iterations")

trained_policy = mccfr_solver.average_policy()
print("\nTraining complete! Running step-by-step game simulation...\n")

# 3. Iterative Playout (No recursion overhead)
state = game.new_initial_state()
player_cards = {}
public_card = None

while not state.is_terminal():
    # Chance Node (Card Dealing)
    if state.is_chance_node():
        outcomes = state.chance_outcomes()
        actions, probs = zip(*outcomes)

        chosen_action = random.choices(actions, weights=probs)[0]
        card_label = CARD_NAMES.get(chosen_action, f"Card {chosen_action}")

        if len(player_cards) < 3:
            p_idx = len(player_cards)
            player_cards[p_idx] = card_label
            print(f"🎲 CHANCE: Dealt {card_label} to Player {p_idx}")
        else:
            public_card = card_label
            print(f"🎲 CHANCE: Flop revealed {card_label}")

        state.apply_action(chosen_action)

    # Decision Node (Player Turn)
    else:
        curr_player = state.current_player()

        # Extract action probabilities recommended by the learned policy
        state_policy = trained_policy.action_probabilities(state)
        actions = list(state_policy.keys())
        probabilities = list(state_policy.values())

        strat_display = ", ".join([f"{ACTION_NAMES[a]}: {p:.1%}" for a, p in state_policy.items()])
        print(f"👤 Player {curr_player} (Holding: {player_cards.get(curr_player, 'Unknown')} | Flop: {public_card or 'None'})")
        print(f"   Learned Strategy Probabilities -> [{strat_display}]")

        # Pick action according to trained probabilities
        chosen_action = random.choices(actions, weights=probabilities)[0]
        print(f"   -> Executed Action: {ACTION_NAMES[chosen_action]}\n")

        state.apply_action(chosen_action)

# Terminal State
returns = state.returns()
payoff_str = ", ".join([f"Player {i}: {returns[i]:+.1f}" for i in range(3)])
print(f"🏁 GAME OVER | Payoffs -> {payoff_str}")

Training 3-player MCCFR over 5,000 iterations (takes ~2-3 seconds)...


KeyboardInterrupt: 

In [5]:
import random
import pyspiel
from open_spiel.python.algorithms import cfr

# Mappings for human readability
CARD_NAMES = {
    0: "Jack ♠", 1: "Jack ♥",
    2: "Queen ♠", 3: "Queen ♥",
    4: "King ♠", 5: "King ♥"
}
ACTION_NAMES = {0: "Fold", 1: "Call / Check", 2: "Raise"}

# 1. Load Game and Train CFR Solver
game = pyspiel.load_game("leduc_poker")
cfr_solver = cfr.CFRSolver(game)

print("Training CFR algorithm over 2,000 iterations...")
for _ in range(2000):
    cfr_solver.evaluate_and_update_policy()

# Extract the converged average policy
trained_policy = cfr_solver.average_policy()
print("Training complete! Playing a game using learned equilibrium strategies.\n")


# 2. Function to Play a Game Using the Trained Policy
def play_trained_game(state, player_cards=None, public_card=None):
    if player_cards is None:
        player_cards = {}

    # Terminal State
    if state.is_terminal():
        returns = state.returns()
        print(f"🏁 GAME OVER | Payoffs -> Player 0: {returns[0]:+.1f}, Player 1: {returns[1]:+.1f}\n")
        return

    # Chance Node (Card Dealing)
    if state.is_chance_node():
        outcomes = state.chance_outcomes()
        actions, probs = zip(*outcomes)

        # Sample card dealing according to true probability
        chosen_action = random.choices(actions, weights=probs)[0]
        card_label = CARD_NAMES[chosen_action]

        if len(player_cards) == 0:
            player_cards[0] = card_label
            print(f"🎲 CHANCE: Dealt {card_label} to Player 0")
        elif len(player_cards) == 1:
            player_cards[1] = card_label
            print(f"🎲 CHANCE: Dealt {card_label} to Player 1")
        else:
            public_card = card_label
            print(f"🎲 CHANCE: Flop revealed {card_label}")

        state.apply_action(chosen_action)
        play_trained_game(state, player_cards, public_card)
        return

    # Decision Node (Player Turn using CFR Policy)
    curr_player = state.current_player()
    info_state_str = state.information_state_string(curr_player)

    # Get action probabilities recommended by the trained CFR policy
    state_policy = trained_policy.action_probabilities(state)

    actions = list(state_policy.keys())
    probabilities = list(state_policy.values())

    # Display the learned strategy breakdown for this turn
    strat_display = ", ".join([f"{ACTION_NAMES[a]}: {p:.1%}" for a, p in state_policy.items()])
    print(f"👤 Player {curr_player} (Holding: {player_cards[curr_player]} | Flop: {public_card or 'None'})")
    print(f"   Learned Strategy Probabilities -> [{strat_display}]")

    # Sample an action weighted by the CFR probabilities
    chosen_action = random.choices(actions, weights=probabilities)[0]
    print(f"   -> Executed Action: {ACTION_NAMES[chosen_action]}\n")

    state.apply_action(chosen_action)
    play_trained_game(state, player_cards, public_card)


# Run a simulation using trained decisions
play_trained_game(game.new_initial_state())


Training CFR algorithm over 2,000 iterations...
Training complete! Playing a game using learned equilibrium strategies.

🎲 CHANCE: Dealt Jack ♥ to Player 0
🎲 CHANCE: Dealt Jack ♠ to Player 1
👤 Player 0 (Holding: Jack ♥ | Flop: None)
   Learned Strategy Probabilities -> [Call / Check: 92.1%, Raise: 7.9%]
   -> Executed Action: Call / Check

👤 Player 1 (Holding: Jack ♠ | Flop: None)
   Learned Strategy Probabilities -> [Call / Check: 70.5%, Raise: 29.5%]
   -> Executed Action: Call / Check

🎲 CHANCE: Flop revealed Queen ♠
👤 Player 0 (Holding: Jack ♥ | Flop: Queen ♠)
   Learned Strategy Probabilities -> [Call / Check: 80.6%, Raise: 19.4%]
   -> Executed Action: Call / Check

👤 Player 1 (Holding: Jack ♠ | Flop: Queen ♠)
   Learned Strategy Probabilities -> [Call / Check: 92.9%, Raise: 7.1%]
   -> Executed Action: Call / Check

🏁 GAME OVER | Payoffs -> Player 0: +0.0, Player 1: +0.0



In [6]:
import numpy as np
import pyspiel
from open_spiel.python import policy
from open_spiel.python.algorithms import cfr

# 1. Load 2-Player Leduc Poker
game = pyspiel.load_game("leduc_poker")

# 2. Train GTO Baseline Policy via CFR
print("Training GTO baseline via CFR...")
cfr_solver = cfr.CFRSolver(game)
for _ in range(100):
    cfr_solver.evaluate_and_update_policy()
gto_policy = cfr_solver.average_policy()


# 3. Parametric Archetype Opponent (Blends GTO with Pure Archetype)
class ParametricArchetypePolicy(policy.Policy):
    def __init__(self, game, player_id, gto_policy, mode="caller", alpha=0.7):
        super().__init__(game, [player_id])
        self.player_id = player_id
        self.gto_policy = gto_policy
        self.mode = mode
        self.alpha = alpha  # Weight on archetype (0.7 = 70% archetype, 30% GTO)

    def action_probabilities(self, state, player_id=None):
        legal_actions = state.legal_actions()
        gto_probs = self.gto_policy.action_probabilities(state)

        # Compute deterministic archetype choice
        archetype_probs = {a: 0.0 for a in legal_actions}
        if self.mode == "caller":
            act = 1 if 1 in legal_actions else 0
            archetype_probs[act] = 1.0
        elif self.mode == "folder":
            act = 0 if 0 in legal_actions else 1
            archetype_probs[act] = 1.0
        elif self.mode == "raiser":
            act = 2 if 2 in legal_actions else (1 if 1 in legal_actions else 0)
            archetype_probs[act] = 1.0
        elif self.mode == "chaotic":
            for a in legal_actions:
                archetype_probs[a] = 1.0 / len(legal_actions)

        # Blend archetype with GTO baseline
        blended_probs = {}
        for a in legal_actions:
            p_gto = gto_probs.get(a, 0.0)
            p_arch = archetype_probs.get(a, 0.0)
            blended_probs[a] = self.alpha * p_arch + (1 - self.alpha) * p_gto

        return blended_probs


# 4. Bayesian Adaptive Agent
class BayesianAdaptiveAgent:
    def __init__(self, game, candidate_profiles):
        self.game = game
        self.profiles = candidate_profiles
        self.profile_names = list(candidate_profiles.keys())
        self.reset_beliefs()

    def reset_beliefs(self):
        """Reset belief distribution to uniform prior (25% each)."""
        num_p = len(self.profile_names)
        self.beliefs = {name: 1.0 / num_p for name in self.profile_names}

    def observe_action(self, state, action_taken):
        """Bayesian update step upon observing opponent action."""
        likelihoods = {}
        total_l = 0.0

        for name, profile_bot in self.profiles.items():
            probs = profile_bot.action_probabilities(state)
            p_act = probs.get(action_taken, 1e-4)  # Small safety floor
            post = p_act * self.beliefs[name]
            likelihoods[name] = post
            total_l += post

        if total_l > 0:
            for name in self.profile_names:
                self.beliefs[name] = likelihoods[name] / total_l


# 5. Build Realistic Opponent Profiles (70% Archetype, 30% GTO)
profiles = {
    "folder": ParametricArchetypePolicy(game, 1, gto_policy, mode="folder", alpha=0.7),
    "caller": ParametricArchetypePolicy(game, 1, gto_policy, mode="caller", alpha=0.7),
    "raiser": ParametricArchetypePolicy(game, 1, gto_policy, mode="raiser", alpha=0.7),
    "chaotic": ParametricArchetypePolicy(game, 1, gto_policy, mode="chaotic", alpha=0.7),
}


# 6. Simulation Function: Tracks Classification Accuracy Across Hands
def evaluate_realization_speed(target_mode, total_hands=30, num_runs=100):
    opponent_bot = profiles[target_mode]
    agent = BayesianAdaptiveAgent(game, profiles)

    belief_history = np.zeros(total_hands)

    for run in range(num_runs):
        agent.reset_beliefs()

        for hand in range(total_hands):
            state = game.new_initial_state()

            while not state.is_terminal():
                if state.is_chance_node():
                    outcomes, probs = zip(*state.chance_outcomes())
                    act = np.random.choice(outcomes, p=probs)
                    state.apply_action(act)

                elif state.current_player() == 0:
                    # Agent samples standard legal actions
                    legal_actions = state.legal_actions()
                    act = np.random.choice(legal_actions)
                    state.apply_action(act)

                else:
                    # Opponent takes turn based on blended strategy
                    opp_probs = opponent_bot.action_probabilities(state)
                    actions = list(opp_probs.keys())
                    p_dist = list(opp_probs.values())
                    act = np.random.choice(actions, p=p_dist)

                    # Agent updates Bayesian belief upon observing opponent move
                    agent.observe_action(state, act)
                    state.apply_action(act)

            # Record running average confidence in true profile
            belief_history[hand] += agent.beliefs[target_mode] / num_runs

    return belief_history


# 7. Print Diagnostics
checkpoints = [1, 2, 5, 10, 15, 20, 30]

print("\nRunning Bayesian inference simulations across 100 iterations per profile...\n")
results = {mode: evaluate_realization_speed(mode) for mode in profiles.keys()}

print("=" * 80)
print("Confidence Level in True Opponent Profile (%) Across Observed Hands")
print("=" * 80)
print(f"{'Hands Observed':<15} | " + " | ".join([f"Vs. {p.capitalize():<11}" for p in profiles.keys()]))
print("-" * 80)

for cp in checkpoints:
    row = [f"{results[mode][cp-1] * 100:11.1f}%" for mode in profiles.keys()]
    print(f"{f'Hand {cp}':<15} | " + " | ".join(row))

print("=" * 80)

Training GTO baseline via CFR...

Running Bayesian inference simulations across 100 iterations per profile...

Confidence Level in True Opponent Profile (%) Across Observed Hands
Hands Observed  | Vs. Folder      | Vs. Caller      | Vs. Raiser      | Vs. Chaotic    
--------------------------------------------------------------------------------
Hand 1          |        48.8% |        47.1% |        52.0% |        28.0%
Hand 2          |        61.9% |        64.3% |        72.9% |        43.0%
Hand 5          |        84.5% |        88.2% |        88.8% |        73.5%
Hand 10         |        95.6% |        96.9% |        95.6% |        89.9%
Hand 15         |        99.0% |        99.1% |        98.9% |        97.5%
Hand 20         |        99.4% |       100.0% |       100.0% |        98.4%
Hand 30         |       100.0% |       100.0% |       100.0% |       100.0%
